# A5: Pandas Analysis (MP1 Patient Satisfaction Dataset)

This notebook loads my **MP1 dataset** (`patient_satisfaction_dataset.csv`) and uses pandas to answer **three analytical questions**.

**Dataset:** Kaggle patient satisfaction survey (452 patients after cleaning, 17 rating columns)

**Class pandas operations used:** `df.head()`, `df.info()`, `df.isnull().sum()`, `df['column'].value_counts()`, `df[df['column'] > value]`, `df.groupby('column')['other'].mean()`

In [ ]:
import pandas as pd

print("pandas version:", pd.__version__)

## Data profile

In [ ]:
# Load the MP1 CSV into a DataFrame
df = pd.read_csv("patient_satisfaction_dataset.csv")

# The original survey used a confusing 1-5 order, so I recode every column to standard order:
# 1 = very unsatisfied ... 5 = very satisfied
RECODE_TO_ORDERED = {
    5: 1,
    2: 2,
    3: 3,
    1: 4,
    4: 5,
}
for col in df.columns:
    df[col] = df[col].map(RECODE_TO_ORDERED)

df = df.dropna()
print("Shape after cleaning:", df.shape)

In [ ]:
# I want to see what a few patient rows look like before I analyze anything.
# head() shows the first rows so I can confirm column names and rating values make sense after recoding.
df.head()

In [ ]:
# I want to know how big the dataset is and whether any columns have missing values.
# info() tells me row count, column count, and non-null counts per column.
df.info()

In [ ]:
# I want to count missing values per column.
# isnull().sum() tells me whether I need dropna() before grouping or filtering.
df.isnull().sum()

## Research Question 1

**What are the most common overall satisfaction ratings (`satisfaction in RM`)?**

In [ ]:
# I want to know which overall satisfaction scores appear most often.
# value_counts() tells me how many patients gave each rating, so I can see whether responses cluster at satisfied vs unsatisfied.
df["satisfaction in RM"].value_counts().sort_index()

### RQ1 interpretation

After recoding, the most common overall satisfaction scores are 2 (not satisfied) and 3 (neutral). Very few patients rate 4 (satisfied) in this cleaned sample. That tells me the dataset is not mostly "happy patients," so I need to look at aspect ratings within realistic satisfaction groups instead of assuming everyone is satisfied.

## Research Question 2

**Among highly satisfied patients, do personable ratings average higher than operational ratings?**

In [ ]:
# I want to focus only on patients with overall satisfaction >= 4.
# This filter keeps rows where patients reported satisfied or very satisfied overall.
high_sat = df[df["satisfaction in RM"] >= 4]

personable_cols = [
    "Communication with dr",
    "friendly health care workers",
    "Quality/experience dr.",
]
operational_cols = [
    "Time waiting",
    "Admin procedures",
    "waiting rooms",
    "parking, playing rooms, caffes",
]

comparison = pd.Series({
    "personable_mean": high_sat[personable_cols].mean().mean(),
    "operational_mean": high_sat[operational_cols].mean().mean(),
    "high_sat_patient_count": len(high_sat),
})
comparison.round(3)

### RQ2 interpretation

Only 7 patients rate overall satisfaction at 4 or higher in this sample, so this is a small group. In that group, operational factors average slightly higher than personable factors. I would not over-interpret this because the filtered group is so small, but the filter still shows how to compare two aspect groups within one satisfaction subset.

## Research Question 3

**How does average doctor communication vary by overall satisfaction level?**

In [ ]:
# I want to compare Communication with dr across each overall satisfaction level.
# groupby('satisfaction in RM').mean() gives the average communication score per satisfaction group.
df.groupby("satisfaction in RM")["Communication with dr"].mean().round(2)

### RQ3 interpretation

Average `Communication with dr` is lower when overall satisfaction is 2 (about 2.85) and higher when overall satisfaction is 3 (about 3.41). The group at satisfaction level 4 is very small, so its mean is less stable. Overall, better communication ratings tend to appear with neutral or higher overall satisfaction, which matches my MP1 focus on doctor communication.